# edge-compress on Kaggle (GPU T4 x2)

Settings: **Accelerator = GPU T4 x2** (not P100), **Internet = on**. For long runs use
**Save Version → Save & Run All**; it runs in the background, so you can close the tab.

Everything is written under `/kaggle/working/edge-compress`, which becomes this version's output.
`run_matrix.py` skips jobs that already have results, so to resume after a dropped session add the
previous version's output as an input and run the restore cell.

In [ ]:
REPO = "https://github.com/<user>/edge-compress.git"   # this repository
%cd /kaggle/working
!git clone -q $REPO || (cd edge-compress && git pull -q)
%cd /kaggle/working/edge-compress
!pip install -q torchao torch-pruning imagehash
!nvidia-smi -L
import torch, torchao; print(torch.__version__, torchao.__version__, torch.cuda.device_count(), "GPUs")

If `import torchao` or the PT2E imports fail, the latest torchao needs a newer torch than the image ships:
`!pip install -q -U torch torchvision torchao` then restart the kernel.

In [ ]:
# Optional: restore checkpoints/results from a previous version's output (add it as an input first)
import glob, shutil, os
prev = glob.glob("/kaggle/input/*/edge-compress")
if prev:
    for d in ("checkpoints", "results", "logits", "RESULTS.md"):
        src = os.path.join(prev[0], d)
        if os.path.isdir(src): shutil.copytree(src, d, dirs_exist_ok=True)
        elif os.path.exists(src): shutil.copy(src, d)
    print("restored from", prev[0])

In [ ]:
# Check that Triton runs on this GPU before relying on it (T4 = sm_75, below Triton's official 8.0+)
!python -m pytest triton_kd -q -x 2>&1 | tail -5

In [ ]:
# Plan, then the CIFAR-100 matrix (teacher + 3 seeds x 9 student configs, scheduled over both GPUs)
!python run_matrix.py --dataset cifar100 --dry-run | grep -c "^\["
!python run_matrix.py --dataset cifar100 2>&1 | tail -60

In [ ]:
!tail -n 30 RESULTS.md
!cat results/cifar100/summary.md

## Drone case study

Add the Kaggle **Bird vs Drone** dataset as an input. Its license governs redistribution of the derived crops.
`drone_data.py` builds 64x64 drone/bird/background crops and runs the perceptual-hash leakage check.

In [ ]:
SRC = glob.glob("/kaggle/input/*bird*drone*")[0]   # adjust if the folder is named differently
!python drone_data.py --src $SRC --out data/drone_crops
!python run_matrix.py --dataset drone --data-dir data 2>&1 | tail -40

In [ ]:
# Triton kernel benchmarks (T4 peak is 320 GB/s) and the compiler-stage dump
!python triton_kd/bench.py --dtype fp32
!python triton_kd/bench.py --dtype fp16 --B 4096
!bash triton_kd/dump_ir.sh

In [ ]:
# Stretch: same KD config with the fused Triton loss on the same seed
!python train.py --model resnet18 --teacher r50_ce --epochs 40 --seed 0 --fused-kd --name r18_kd_fused
!python plots.py --dataset cifar100 --curves r18_kd r18_kd_fused --curve-seed 0

In [ ]:
# Bundle what the CPU benchmark machine needs (checkpoints + results + logits)
!tar czf /kaggle/working/edge-compress-artifacts.tgz checkpoints results logits RESULTS.md triton_kd/results
!ls -la /kaggle/working/*.tgz